UNet Implementation

In [1]:
from pathlib import Path

import polars as pl

import torch
from torch import Tensor

In [ ]:
dataset_dir = Path('datasets/nih-chest-xray')

df_test = pl.read_lines(dataset_dir / 'test_list.txt', name='Image Index')
df_train = pl.read_lines(dataset_dir / 'train_val_list.txt', name='Image Index')
df_profile = pl.read_csv(dataset_dir / 'Data_Entry_2017.csv')
df_bbox = pl.read_csv(dataset_dir / 'BBox_List_2017.csv')

df_dir = pl.DataFrame(
    ((uri.name, str(uri)) for uri in dataset_dir.glob('images_0*/images/*.png')),
    schema=['Image Index', 'Image Path'],
    orient='row'
)

train_corpus = df_profile.join(
    df_train,
    on='Image Index',
    how='inner'
)

test_corpus = df_profile.join(
    df_test,
    on='Image Index',
    how='inner'
)

In [21]:
df_dir.head()

Image Index,Image Path
str,str
"""00004808_032.png""","""datasets/nih-chest-xray/images…"
"""00005391_003.png""","""datasets/nih-chest-xray/images…"
"""00006391_005.png""","""datasets/nih-chest-xray/images…"
"""00006386_002.png""","""datasets/nih-chest-xray/images…"
"""00004959_008.png""","""datasets/nih-chest-xray/images…"


In [15]:
ll = dataset_dir.glob('images_0*/images/*.png')

In [16]:
df_dir = pl.DataFrame(
    [(uri.name, str(uri)) for uri in ll],
    schema=['Image Index', 'Image Path'],
    orient='row'
)

df_dir

Image Index,Image Path
str,str
"""00004808_032.png""","""datasets/nih-chest-xray/images…"
"""00005391_003.png""","""datasets/nih-chest-xray/images…"
"""00006391_005.png""","""datasets/nih-chest-xray/images…"
"""00006386_002.png""","""datasets/nih-chest-xray/images…"
"""00004959_008.png""","""datasets/nih-chest-xray/images…"
…,…
"""00000181_062.png""","""datasets/nih-chest-xray/images…"
"""00001308_000.png""","""datasets/nih-chest-xray/images…"
"""00000885_000.png""","""datasets/nih-chest-xray/images…"


In [15]:
test_corpus.head()

Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],
str,str,i64,i64,i64,str,str,i64,i64,f64,f64,str
"""00000003_000.png""","""Hernia""",0,3,81,"""F""","""PA""",2582,2991,0.143,0.143,null
"""00000003_001.png""","""Hernia""",1,3,74,"""F""","""PA""",2500,2048,0.168,0.168,null
"""00000003_002.png""","""Hernia""",2,3,75,"""F""","""PA""",2048,2500,0.168,0.168,null
"""00000003_003.png""","""Hernia|Infiltration""",3,3,76,"""F""","""PA""",2698,2991,0.143,0.143,null
"""00000003_004.png""","""Hernia""",4,3,77,"""F""","""PA""",2500,2048,0.168,0.168,null


In [13]:
df_bbox.head()

Image Index,Finding Label,Bbox [x,y,w,h],,_duplicated_0,_duplicated_1
str,str,f64,f64,f64,f64,str,str,str
"""00013118_008.png""","""Atelectasis""",225.084746,547.019217,86.779661,79.186441,null,null,null
"""00014716_007.png""","""Atelectasis""",686.101695,131.543498,185.491525,313.491525,null,null,null
"""00029817_009.png""","""Atelectasis""",221.830508,317.053115,155.118644,216.949153,null,null,null
"""00014687_001.png""","""Atelectasis""",726.237288,494.95142,141.016949,55.322034,null,null,null
"""00017877_001.png""","""Atelectasis""",660.067797,569.780787,200.677966,78.101695,null,null,null
